In [1]:
!pip install sentence-transformers


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
import minsearch

# 1. Cargamos los documentos locales
with open('documents-with-ids.json', 'rt') as f_in:
    documents = json.load(f_in)

# 2. Cargamos nuestro Ground Truth y filtramos SOLO el curso de Machine Learning
df_ground_truth = pd.read_csv('ground-truth-data.csv')
df_ground_truth = df_ground_truth[df_ground_truth.course == 'machine-learning-zoomcamp']
ground_truth = df_ground_truth.to_dict(orient='records')

print(f"Total de preguntas a evaluar (solo ML): {len(ground_truth)}")

# 3. Descargamos el modelo de embeddings para convertir texto a vectores
print("Cargando el modelo de Sentence Transformers...")
model_name = 'multi-qa-MiniLM-L6-cos-v1'
model = SentenceTransformer(model_name)

# 4. Convertimos nuestros documentos a vectores
print("Creando vectores para los documentos (Esto tomará un par de minutos)...")
vectors = []
for doc in tqdm(documents):
    question = doc['question']
    text = doc['text']
    # Unimos la pregunta y la respuesta para tener más contexto
    vector = model.encode(question + ' ' + text)
    vectors.append(vector)

vectors = np.array(vectors)

# 5. Guardamos los vectores en el motor de minsearch
print("Indexando en minsearch VectorSearch...")
vindex = minsearch.vector.VectorSearch(keyword_fields=['course'])
vindex.fit(vectors, documents)

print("¡Motor de búsqueda vectorial listo!")

Total de preguntas a evaluar (solo ML): 1830
Cargando el modelo de Sentence Transformers...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Creando vectores para los documentos (Esto tomará un par de minutos)...


  0%|          | 0/948 [00:00<?, ?it/s]

Indexando en minsearch VectorSearch...
¡Motor de búsqueda vectorial listo!


In [3]:
from openai import OpenAI

client = OpenAI()

# 1. Funciones de Búsqueda Vectorial
def minsearch_vector_search(vector, course):
    return vindex.search(
        vector,
        filter_dict={'course': course},
        num_results=5
    )

def question_text_vector(q):
    question = q['question']
    course = q['course']
    
    # Convertimos la pregunta a vector matemático
    v_q = model.encode(question)
    
    return minsearch_vector_search(v_q, course)

# 2. Función para armar el Prompt
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT: 
{context}
""".strip()

    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    
    prompt = prompt_template.format(question=query, context=context).strip()
    return prompt

# 3. Función para consultar al LLM (OpenAI)
def llm(prompt, model='gpt-4o-mini'):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# 4. Función RAG maestra
def rag(query: dict, model='gpt-4o-mini') -> str:
    search_results = question_text_vector(query)
    prompt = build_prompt(query['question'], search_results)
    answer = llm(prompt, model=model)
    return answer

# --- PRUEBA DEL SISTEMA ---
# Tomamos la pregunta número 11 (índice 10) de nuestra lista
pregunta_prueba = ground_truth[10]

print(f"Pregunta original: {pregunta_prueba['question']}")
print("Buscando y generando respuesta...\n")

respuesta_generada = rag(pregunta_prueba)

print(f"Respuesta del LLM:\n{respuesta_generada}")

Pregunta original: Are sessions recorded if I miss one?
Buscando y generando respuesta...

Respuesta del LLM:
Yes, all sessions are recorded, so if you miss one, you won't miss anything. You can also ask questions for office hours in advance, which will be covered during the live stream. Additionally, you can ask questions in Slack.


In [4]:
# 1. Creamos el índice de documentos usando sus IDs (¡La línea que faltaba!)
doc_idx = {d['id']: d for d in documents}

# 2. Obtenemos la respuesta original (la oficial del FAQ)
documento_oficial = doc_idx[pregunta_prueba['document']]
respuesta_original = documento_oficial['text']

print("--- COMPARACIÓN ---")
print(f"Respuesta Original:\n{respuesta_original}\n")
print(f"Respuesta del LLM:\n{respuesta_generada}\n")

# 3. Convertimos ambas respuestas a vectores matemáticos
v_llm = model.encode(respuesta_generada)
v_orig = model.encode(respuesta_original)

# 4. Calculamos la Similitud del Coseno (Producto punto)
similitud = v_llm.dot(v_orig)

print(f"CALIFICACIÓN DE SIMILITUD (Cosine Similarity): {similitud:.4f}")

--- COMPARACIÓN ---
Respuesta Original:
Everything is recorded, so you won’t miss anything. You will be able to ask your questions for office hours in advance and we will cover them during the live stream. Also, you can always ask questions in Slack.

Respuesta del LLM:
Yes, all sessions are recorded, so if you miss one, you won't miss anything. You can also ask questions for office hours in advance, which will be covered during the live stream. Additionally, you can ask questions in Slack.

CALIFICACIÓN DE SIMILITUD (Cosine Similarity): 0.8475


In [5]:
!pip install seaborn
import os
import urllib.request
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Descargamos el archivo con las 1830 respuestas ya generadas por GPT-4o
# url_csv = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/03-evaluation/data/results-gpt4o.csv'
url_csv = 'https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/refs/heads/main/cohorts/2025/03-evaluation/rag_evaluation/data/results-gpt4o.csv'

if not os.path.exists('results-gpt4o.csv'):
    print("Descargando respuestas pre-generadas (results-gpt4o.csv)...")
    urllib.request.urlretrieve(url_csv, 'results-gpt4o.csv')

df_gpt4o = pd.read_csv('results-gpt4o.csv')
resultados_gpt4o = df_gpt4o.to_dict(orient='records')

# 2. Función para calcular similitud de un registro completo
def compute_similarity(record):
    answer_orig = record['answer_orig']
    answer_llm = record['answer_llm']
    
    v_llm = model.encode(answer_llm)
    v_orig = model.encode(answer_orig)
    
    return v_llm.dot(v_orig)

# 3. Aplicamos la evaluación a los 1830 documentos
print("Calificando las 1830 respuestas (Esto tomará 1-2 minutos)...")
similarity = []
for record in tqdm(resultados_gpt4o):
    sim = compute_similarity(record)
    similarity.append(sim)

# 4. Agregamos las calificaciones a la tabla y mostramos el resumen estadístico
df_gpt4o['cosine'] = similarity
print("\n--- RESUMEN DE CALIFICACIONES (GPT-4o) ---")
print(df_gpt4o['cosine'].describe())

# (Opcional) Guardamos nuestra tabla ya calificada
df_gpt4o.to_csv('results-gpt4o-cosine.csv', index=False)


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Calificando las 1830 respuestas (Esto tomará 1-2 minutos)...


  0%|          | 0/1830 [00:00<?, ?it/s]


--- RESUMEN DE CALIFICACIONES (GPT-4o) ---
count    1830.000000
mean        0.679129
std         0.217995
min        -0.153426
25%         0.591460
50%         0.734788
75%         0.835390
max         0.995338
Name: cosine, dtype: float64


In [6]:
import json

# 1. La plantilla estricta para nuestro Juez
prompt_juez = """
You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.
Your task is to analyze the relevance of the generated answer compared to the original answer provided.
Based on the relevance and similarity of the generated answer to the original answer, you will classify
it as "NON_RELEVANT", "PARTLY_RELEVANT", or "RELEVANT".

Here is the data for evaluation:

Original Answer: {answer_orig}
Generated Question: {question}
Generated Answer: {answer_llm}

Please analyze the content and context of the generated answer in relation to the original
answer and provide your evaluation in parsable JSON without using code blocks:

{{
  "Relevance": "NON_RELEVANT" | "PARTLY_RELEVANT" | "RELEVANT",
  "Explanation": "[Provide a brief explanation for your evaluation]"
}}
""".strip()

# 2. Tomamos una muestra aleatoria de 150 preguntas para evaluar
df_sample = df_gpt4o.sample(n=150, random_state=1)
samples = df_sample.to_dict(orient='records')

print("El Juez (LLM) está calificando 150 respuestas. Esto tomará un minuto...")

# 3. Ponemos al juez a trabajar
evaluaciones = []
for record in tqdm(samples):
    # Rellenamos la plantilla con los datos
    prompt = prompt_juez.format(**record)
    # Llamamos a OpenAI
    evaluation = llm(prompt, model='gpt-4o-mini')
    evaluaciones.append(evaluation)

# 4. Limpiamos las respuestas (por si el LLM escribe mal el JSON)
json_evaluaciones = []
for str_eval in evaluaciones:
    try:
        json_eval = json.loads(str_eval)
        json_evaluaciones.append(json_eval)
    except json.JSONDecodeError:
        print("Saltando una respuesta mal formateada...")
        continue

# 5. Mostramos el boletín de calificaciones final
df_evaluaciones = pd.DataFrame(json_evaluaciones)
print("\n--- NOTAS DEL JUEZ (LLM-as-a-Judge) ---")
print(df_evaluaciones['Relevance'].value_counts())

El Juez (LLM) está calificando 150 respuestas. Esto tomará un minuto...


  0%|          | 0/150 [00:00<?, ?it/s]


--- NOTAS DEL JUEZ (LLM-as-a-Judge) ---
Relevance
RELEVANT           125
PARTLY_RELEVANT     18
NON_RELEVANT         7
Name: count, dtype: int64
